# 🔧 Lesson 08c — Evaluation Harness & Cost Tracker
**Duration:** ~1 hour | Part of Capstone Preparation

In [ ]:
# ── Shared imports (run this first) ──────────────────────────────
import os, json, time, datetime, re
from pathlib import Path
from typing import Optional, List
from pydantic import BaseModel, Field, field_validator
from datasets import load_dataset
import pandas as pd
from openai import OpenAI
from anthropic import Anthropic
from llm_checker import check, hint, evaluate, progress, show_exercise

openai_client = OpenAI()
claude_client = Anthropic()
lm_client     = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

Path("data/capstone").mkdir(parents=True, exist_ok=True)
print("✅  Module 8 imports ready")

---
## 08c — Evaluation Harness & Cost Tracker

**Goal:** Build the golden eval set, run RAGAS on policy Q&A, set up the LLM-as-judge for retention offers, and wire cost tracking into every LLM call.


In [ ]:
# ── Cost tracker ──────────────────────────────────────────────────
from dataclasses import dataclass, field

@dataclass
class CostTracker:
    total_usd: float = 0.0
    calls:     int   = 0
    log:       list  = field(default_factory=list)

    PRICES = {
        "gpt-4o":                    {"input": 0.005,  "output": 0.015},
        "gpt-4o-mini":               {"input": 0.00015,"output": 0.0006},
        "claude-3-5-sonnet-20241022":{"input": 0.003,  "output": 0.015},
        "claude-3-7-sonnet-20250219":{"input": 0.003,  "output": 0.015},
        "local-model":               {"input": 0.0,    "output": 0.0},
    }

    def record(self, model: str, in_tok: int, out_tok: int, label: str = ""):
        p = self.PRICES.get(model, {"input": 0.002, "output": 0.008})
        cost = (in_tok * p["input"] + out_tok * p["output"]) / 1000
        self.total_usd += cost
        self.calls     += 1
        self.log.append({"model":model,"in_tok":in_tok,"out_tok":out_tok,
                         "cost_usd":cost,"label":label,
                         "ts":datetime.datetime.now().isoformat(timespec="seconds")})
        return cost

    def summary(self):
        print(f"\n💰 Cost Summary: {self.calls} calls | ${self.total_usd:.4f} total")
        print(f"   Projection → 10,000 customers/month: ${self.total_usd * 100:.2f}/month")

tracker = CostTracker()
print("✅ Cost tracker ready")

In [ ]:
# ── Build golden eval set ─────────────────────────────────────────
GOLDEN_EVAL = []

# 10 policy Q&A pairs
policy_topics_sample = POLICY_TOPICS[:10]
for topic in policy_topics_sample:
    context = retrieve_policy(topic, n_results=1)
    question = f"What is the bank's policy on {topic}?"
    # Generate reference answer
    msg = lm_client.chat.completions.create(
        model="local-model", max_tokens=150,
        messages=[{"role":"user","content":f"Context: {context}\nQuestion: {question}\nAnswer in 1-2 sentences."}]
    )
    tracker.record("local-model", 200, msg.usage.completion_tokens if hasattr(msg.usage,"completion_tokens") else 150, "golden_eval_build")
    GOLDEN_EVAL.append({
        "type":      "policy_qa",
        "question":  question,
        "context":   context,
        "reference": msg.choices[0].message.content.strip(),
    })

print(f"✅ Golden eval set: {len(GOLDEN_EVAL)} policy Q&A pairs")
print("Sample:", GOLDEN_EVAL[0]["question"])

In [ ]:
# ── LLM-as-judge for retention offers ────────────────────────────

JUDGE_RUBRIC = """
Rate the following bank retention offer on 3 dimensions (1-5 each):
1. Clarity: Is it clear what the customer receives?
2. Relevance: Is it tailored to the customer's apparent risk?
3. Compliance_safe: Does it avoid legally unsafe promises?

Reply ONLY with JSON: {clarity: int, relevance: int, compliance_safe: int, overall: float, notes: str}
"""

def judge_offer(offer_text: str, customer_context: str = "") -> dict:
    """Score a retention offer with LLM-as-judge."""
    prompt = f"""CUSTOMER CONTEXT: {customer_context or 'High churn risk, 3-year customer, $80/month'}

RETENTION OFFER: {offer_text}

{JUDGE_RUBRIC}"""
    msg = lm_client.chat.completions.create(
        model="local-model", max_tokens=200,
        messages=[{"role":"user","content": prompt}]
    )
    tracker.record("local-model", 250, 200, "judge_offer")
    try:
        raw = msg.choices[0].message.content.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
        return json.loads(raw)
    except Exception:
        return {"clarity":3,"relevance":3,"compliance_safe":3,"overall":3.0,"notes":"parse error"}

# Test the judge
test_offer = "We'd like to offer you 3 months of free premium banking as a valued customer."
score = judge_offer(test_offer)
print("Judge score:", score)

In [ ]:
# ── Run RAGAS-style evaluation on policy Q&A ─────────────────────
# (Simplified RAGAS without the full framework for compatibility)

def ragas_faithfulness(question: str, context: str, answer: str) -> float:
    """Score 0-1: does the answer only use information from the context?"""
    msg = lm_client.chat.completions.create(
        model="local-model", max_tokens=100,
        messages=[{"role":"user","content":
            f"Question: {question}\nContext: {context[:500]}\nAnswer: {answer}\n"
            "Score 0.0-1.0: does the answer rely ONLY on the context (1=fully grounded, 0=hallucinated). "
            "Reply with just a number."}]
    )
    tracker.record("local-model", 300, 50, "ragas_faithfulness")
    try:
        return float(re.findall(r"\d+\.?\d*", msg.choices[0].message.content)[0])
    except Exception:
        return 0.5

# Evaluate 5 policy pairs
eval_scores = []
for item in GOLDEN_EVAL[:5]:
    answer = retrieve_policy(item["question"], n_results=1)
    score  = ragas_faithfulness(item["question"], item["context"], answer)
    eval_scores.append(score)
    print(f"  Faithfulness: {score:.2f} | Q: {item['question'][:60]}")

avg_faithfulness = sum(eval_scores) / len(eval_scores)
print(f"\n✅ Avg RAGAS Faithfulness: {avg_faithfulness:.2f}")
tracker.summary()

In [ ]:
# ── Save golden eval set ──────────────────────────────────────────
eval_path = Path("data/capstone/golden_eval.jsonl")
with open(eval_path, "w") as f:
    for item in GOLDEN_EVAL:
        f.write(json.dumps(item) + "\n")
print(f"✅ Golden eval set saved → {eval_path}")

check([
    (len(GOLDEN_EVAL) >= 10,        "≥ 10 golden eval items"),
    (eval_path.exists(),             "golden_eval.jsonl saved"),
    (avg_faithfulness >= 0.5,       f"Avg RAGAS faithfulness ≥ 0.5 (got {avg_faithfulness:.2f})"),
    (tracker.total_usd >= 0,        "Cost tracker functional"),
    (len(tracker.log) >= 5,         "Cost log has ≥ 5 entries"),
], exercise_id="08c-1")

In [ ]:
# ── Module 8 Progress ─────────────────────────────────────────────
progress()

---
## ✅ Module 8 Integration Readiness

Before opening the Capstone notebook, confirm:
- ☐ `complaints_extracted.jsonl` — 100 CFPB complaints extracted (pass rate ≥ 90%)
- ☐ ChromaDB collection — ≥ 30 policy documents indexed
- ☐ XGBoost model — AUC ≥ 0.68 on Telco Churn
- ☐ `hybrid_routing_log.jsonl` — hybrid router tested
- ☐ `golden_eval.jsonl` — 10+ Q&A pairs ready
- ☐ Cost tracker — recording all LLM calls

**You are now ready for the Capstone! →** `capstone_banking_crm.ipynb`
